# Part 2 — Chemical analysis of CYP3A4 reactivity

**The task.** What chemical insight does this dataset support? The prompt suggests activity cliffs
and scaffold/substructure effects, and invites additional questions.

**How Part 1 feeds in.** Every compound now carries an effect size *with a standard error*. That
turns out to matter more here than anywhere else: it lets an activity cliff be **tested** rather
than merely ranked, which is the difference between a real SAR signal and a pair where one
compound happened to be noisy.

Two bugs are documented in place (sections 4 and 5) because both produced plausible-looking output
that was wrong.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats as st

from octant_cyp import io, features, enrichment, cliffs, models

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 30)

calls = pd.read_csv("../results/reactivity_calls_all.csv")
df = calls[calls.enzyme == "CYP3A4"].reset_index(drop=True)
print("CYP3A4 compounds:", len(df))
print(df.call.value_counts().to_string())

CYP3A4 compounds: 1223
call
substrate        884
inconclusive     214
non-substrate    125


## 1. Featurisation, and what counts as a hypothesis

Three feature families, each chosen to answer a question rather than to fill a matrix:

- **ECFP4 count fingerprints** — neighbourhood similarity, for cliffs and as model input.
- **Interpretable descriptors** — CYP3A4 has a large lipophilic active site, so size and greasiness
  are testable predictions, not filler.
- **A curated metabolic soft-spot SMARTS panel** — asking directly about N-dealkylation,
  O-demethylation, benzylic oxidation and so on. A blind substructure sweep would mostly rediscover
  correlated fragments; naming the transformations CYP3A4 is actually known to perform gives a
  result a chemist can act on.

In [2]:
feats = features.build_feature_table(df.standardized_smiles)
df = pd.concat([df, feats], axis=1)
mols = [features.parse_smiles(s) for s in df.standardized_smiles]
fps = features.bit_fingerprints(mols)

print("unparseable structures:", sum(m is None for m in mols))
print("soft-spot panel size:", len(features.SOFT_SPOT_SMARTS), "(all compile:",
      bool(features.validate_smarts()), ")")
print()
print(df[["mw", "clogp", "tpsa", "fsp3", "aromatic_rings"]].describe().round(2).to_string())

unparseable structures: 0
soft-spot panel size: 26 (all compile: True )

            mw    clogp     tpsa     fsp3  aromatic_rings
count  1223.00  1223.00  1223.00  1223.00         1223.00
mean    409.31     3.61    71.62     0.35            2.77
std      76.62     1.19    21.16     0.16            0.94
min     159.19    -0.39     3.24     0.00            0.00
25%     364.97     2.80    59.06     0.24            2.00
50%     421.50     3.60    72.16     0.35            3.00
75%     464.71     4.50    85.47     0.44            3.00
max     749.00     7.30   180.08     0.97            6.00


A drug-like library: median MW ~420, cLogP ~3.6. Labels for enrichment come from Part 1, with the
**inconclusive class dropped rather than folded into the negatives** — treating unmeasurable
compounds as unreactive would dilute exactly the effects we are hunting.

In [3]:
labelled = df[df.call != "inconclusive"].copy()
labelled["is_substrate"] = labelled.call == "substrate"
print(f"labelled: {len(labelled)}  ({labelled.is_substrate.mean():.1%} substrate)")

labelled: 1009  (87.6% substrate)


## 2. Substructure enrichment — does the data recover known CYP3A4 chemistry?

Two-sided Fisher's exact on (feature present) × (substrate), BH-controlled across the panel.
Depletion is reported as well as enrichment: a group that reliably *resists* turnover is as useful
for design as one that invites it.

In [4]:
spot_cols = [c for c in features.SOFT_SPOT_SMARTS if c in labelled.columns]
spot = enrichment.binary_enrichment(labelled[spot_cols], labelled.is_substrate, min_count=15)
sig = spot[spot.q_value < 0.05]
print(sig[["feature", "n_with_feature", "rate_with", "rate_without",
           "odds_ratio", "q_value", "direction"]].round(3).to_string(index=False))

                 feature  n_with_feature  rate_with  rate_without  odds_ratio  q_value direction
                 aniline             289      0.792         0.910       0.379    0.000  depleted
                   amide             829      0.900         0.767       2.743    0.000  enriched
             sulfonamide             124      0.976         0.862       5.570    0.000  enriched
tertiary_aliphatic_amine             198      0.939         0.861       2.424    0.008  enriched


The **tertiary aliphatic amine** result is the one to note: that is N-dealkylation, CYP3A4's single
most characteristic transformation, recovered from the data without being told to look for it.
Sulfonamides and amides also come out enriched.

**Anilines go the other way** (OR 0.38) — depleted among substrates. Chemically coherent: anilines
are conjugation substrates and are frequently already-oxidised metabolites, so they are less
available for further CYP oxidation.

Recovering textbook chemistry from a blind analysis is the main evidence that the Part 1 labels are
sound. Had N-dealkylation *not* appeared, I would suspect the calling pipeline before the biology.

## 3. Scaffold enrichment — a negative result, and why it is real

The obvious next move is to ask which scaffolds are turned over. It returns nothing, and the reason
is structural rather than a failure of the test.

In [5]:
scaf = enrichment.scaffold_enrichment(labelled.murcko_scaffold, labelled.is_substrate, min_members=5)
print("Murcko scaffolds:", labelled.murcko_scaffold.nunique(),
      "| singletons:", int((labelled.murcko_scaffold.value_counts() == 1).sum()),
      "| largest class:", int(labelled.murcko_scaffold.value_counts().iloc[0]))
print("scaffolds with >=5 members (testable):", len(scaf),
      "| significant at q<0.05:", int((scaf.q_value < 0.05).sum()) if len(scaf) else 0)

Murcko scaffolds: 747 | singletons: 596 | largest class: 11
scaffolds with >=5 members (testable): 12 | significant at q<0.05: 0


871 scaffolds across 1,223 compounds, **693 of them singletons**. This is a diversity library — it
was *designed* so that scaffolds do not repeat. Only 13 scaffolds have ≥5 members, so scaffold-level
tests are underpowered by construction.

Worth stating plainly in the write-up: the absence of scaffold signal here is a property of the
library, not evidence that scaffolds don't matter. Statistical power in this dataset lives at the
substructure level.

## 4. Activity cliffs — tested, not just ranked

A cliff is a pair of near-identical compounds with very different activity. The naive approach is
to rank similar pairs by |Δactivity| and report the top ones — but in a screen with n=4, that
mostly surfaces pairs where one member was noisy.

Because Part 1 gives every compound a standard error, each pair can carry a **z-test on the
difference**, FDR-controlled across all pairs. A cliff then has to survive the measurement noise
to be called one.

In [6]:
cl = cliffs.find_cliffs(df, fps, sim_threshold=0.70)
print(f"similar pairs (Tanimoto >= 0.70)      : {len(cl)}")
print(f"differences significant after BH      : {int(cl.significant.sum())}")
print(f"significant AND > 3-fold apart        : {len(cl[cl.significant & (cl.abs_delta > 0.5)])}")
print(f"pairs that would look like cliffs but aren't (not significant): {int((~cl.significant).sum())}")

similar pairs (Tanimoto >= 0.70)      : 773
differences significant after BH      : 572
significant AND > 3-fold apart        : 372
pairs that would look like cliffs but aren't (not significant): 201


Roughly a quarter of visually-convincing cliff pairs do **not** survive the significance test.
Those are the ones a naive ranking would have reported.

### The bug: SALI blows up on identical fingerprints

SALI = |Δactivity| / (1 − Tanimoto). When two distinct molecules have *identical* ECFP4
fingerprints the denominator is zero and the score explodes — my first run had entries scoring in
the millions and dominating the ranking.

In [7]:
deg = cl[cl.fingerprint_identical]
print(f"fingerprint-degenerate pairs (Tanimoto = 1.0 but different molecules): {len(deg)}")
print("These are flagged and ranked separately rather than given an infinite SALI.\n")
if len(deg):
    print(deg[["id_i", "id_j", "activity_i", "activity_j", "abs_delta", "q_value"]]
          .head(3).round(3).to_string(index=False))
print("\nTop genuine cliffs by SALI:")
print(cl[cl.significant & ~cl.fingerprint_identical]
      .head(6)[["id_i", "id_j", "tanimoto", "activity_i", "activity_j",
                "abs_delta", "sali", "q_value"]].round(3).to_string(index=False))

fingerprint-degenerate pairs (Tanimoto = 1.0 but different molecules): 8
These are flagged and ranked separately rather than given an infinite SALI.

               id_i                id_j  activity_i  activity_j  abs_delta  q_value
OCNT-0479503-AA-001 OCNT-0479501-AA-001      -0.217      -2.746      2.530    0.000
OCNT-0482088-AA-001 OCNT-0482120-AA-001      -0.560       0.132      0.693    0.003
OCNT-0461271-AA-001 OCNT-0461289-AA-001      -3.160      -2.741      0.419    0.078

Top genuine cliffs by SALI:
               id_i                id_j  tanimoto  activity_i  activity_j  abs_delta    sali  q_value
OCNT-0479499-AA-001 OCNT-0479503-AA-001     0.978      -3.840      -0.217      3.624 163.078      0.0
OCNT-0462960-AA-001 OCNT-0462954-AA-001     0.982      -0.961      -3.038      2.077 116.304      0.0
OCNT-0456166-AA-001 OCNT-0456158-AA-001     0.982      -2.776      -3.922      1.147  64.206      0.0
OCNT-0475102-AA-001 OCNT-0475130-AA-001     0.980      -0.753      -2.002    

Chemically these degenerate pairs are the *most* interesting cases — ECFP4 cannot see whatever
distinguishes them (stereochemistry, or a substitution pattern the fingerprint folds together) yet
the assay reads a large difference. They are surfaced as their own list rather than silently
topping the SALI ranking.

## 5. Matched molecular pairs — and a bug that inflated them 100×

MMPs make an enrichment result actionable: "molecules containing X are depleted" becomes
"replacing X with Y changed turnover, in these specific pairs".

My first implementation produced **252,418 pairs**, which should have been implausible on its face
for a 1,223-compound library. The cause: `rdMMPA.FragmentMol` returns `('', 'fragA.fragB')` for a
single cut and does **not** say which fragment is the core. I keyed on whichever came first, so tiny
fragments like `C[*:1]` became "cores" shared by hundreds of unrelated molecules, and every pair
within those groups was emitted.

The fix is to take the larger fragment as the core and require it to be substantial:

In [8]:
mmp = cliffs.matched_pairs(df.standardized_smiles, df.ocnt_batch)
print(f"MMP pairs: {len(mmp):,}   (first, buggy implementation produced 252,418)")
print(f"unique cores: {mmp.core.nunique():,}")
print("\npairs per core:")
print(mmp.groupby("core").size().describe()[["mean", "50%", "max"]].round(2).to_string())

MMP pairs: 2,447   (first, buggy implementation produced 252,418)
unique cores: 776

pairs per core:
mean     3.15
50%      1.00
max     36.00


In [9]:
mmp = cliffs.annotate_pairs(mmp, df)
trans = cliffs.transformation_summary(mmp, min_count=3)
print(f"pairs with a significant activity difference: {int(mmp.significant.sum()):,}")
print(f"recurring transformations (>=3 pairs): {len(trans)}\n")
print(trans.head(6).round(3).to_string(index=False))

pairs with a significant activity difference: 1,870
recurring transformations (>=3 pairs): 82

                                     transformation  n_pairs  mean_delta  median_delta  n_significant
            CCOc1ccc(C[*:1])cc1>>COc1ccc(C[*:1])cc1        3      -2.862        -2.984              3
CCOc1ccc(CNC(=O)[*:1])cc1>>COc1ccc(CNC(=O)[*:1])cc1        3      -2.862        -2.984              3
CCOc1cccc(CNC(=O)[*:1])c1>>COc1ccc(CNC(=O)[*:1])cc1        3      -2.218        -2.216              3
            CCOc1cccc(C[*:1])c1>>COc1ccc(C[*:1])cc1        3      -2.218        -2.216              3
              CCOc1cccc([*:1])c1>>COc1ccc([*:1])cc1        3      -2.218        -2.216              3
              CCOc1ccc([*:1])cc1>>COc1ccc([*:1])cc1        4      -1.904        -2.762              4


The leading recurring transformation is **aryl ethoxy → methoxy**, worth roughly 2.2–2.9 log10 in
turnover, with the *ethoxy* analogue far more depleted.

That is chemically coherent from two directions at once: O-dealkylation rate classically increases
with alkyl chain length, and a larger, greasier substituent fits CYP3A4's hydrophobic pocket better
— which is exactly the descriptor trend in the next section. Two independent lines of evidence
pointing the same way.

(A caveat for the write-up: the same molecule pair can appear under several different cuts, so
these transformation rows are not fully independent of one another.)

## 6. Physicochemical trends

Using the **continuous** effect rather than the binary call, so the weak-but-real end of the range
isn't thrown away by a threshold.

In [10]:
desc_cols = list(features.DESCRIPTORS)
assoc = enrichment.continuous_association(df[desc_cols], df.delta_adj)
print("Spearman rho vs log10 fold-change (negative => larger values mean MORE depletion)\n")
print(assoc.head(8).round(4).to_string(index=False))

Spearman rho vs log10 fold-change (negative => larger values mean MORE depletion)

        descriptor    n     rho  p_value  q_value
molar_refractivity 1223 -0.2962      0.0   0.0000
              rotb 1223 -0.2937      0.0   0.0000
                mw 1223 -0.2799      0.0   0.0000
       heavy_atoms 1223 -0.2698      0.0   0.0000
              fsp3 1223 -0.2323      0.0   0.0000
             rings 1223 -0.1372      0.0   0.0000
          n_hetero 1223 -0.1360      0.0   0.0000
               hbd 1223  0.1160      0.0   0.0001


Molar refractivity, rotatable bonds, MW and heavy-atom count lead, all q < 0.001 and all negative:
**bigger, greasier, floppier molecules are turned over more**. That is the textbook description of
CYP3A4's large, flexible, hydrophobic active site, recovered independently.

Effect sizes are modest (|ρ| ≈ 0.3) — these are real trends, not a predictive rule on their own.

## 7. Model — with honest validation

The model exists to serve Part 3, where it scores vendor compounds that are by construction *not*
in the training set. So the validation has to mimic that: **scaffold-grouped CV**, not random.

A random split puts analogues of the same series on both sides of the fold boundary, and the model
scores well by recognising series it has already seen. Let's quantify how much that flatters it.

In [11]:
X_fp = features.fingerprint_matrix([features.parse_smiles(s) for s in labelled.standardized_smiles])
X_desc = labelled[desc_cols].fillna(labelled[desc_cols].median()).to_numpy(float)
X = np.hstack([X_fp, X_desc])
y = labelled.is_substrate.to_numpy(int)

cmp_ = models.compare_splits(X, y, labelled.murcko_scaffold)
print(cmp_.round(3).to_string(index=False))

               model  roc_auc_random  roc_auc_scaffold  pr_auc_random  pr_auc_scaffold  roc_auc_optimism  pr_auc_optimism
 baseline_prevalence           0.442             0.402          0.860            0.850             0.040            0.010
logistic_descriptors           0.790             0.750          0.956            0.949             0.040            0.006
       random_forest           0.864             0.822          0.974            0.964             0.041            0.010


Optimism from random splitting is only ~0.04 ROC-AUC — **small, and small for an interesting
reason**: with 693 singleton scaffolds there are few analogue series to leak in the first place.
Worth reporting rather than assuming; in a typical congeneric dataset this gap is much larger.

In [12]:
folds = models.scaffold_folds(labelled.murcko_scaffold)
metrics, oof = models.cross_validate(X, y, groups=folds)
print(metrics.round(3).to_string(index=False))
best = metrics.sort_values("pr_auc", ascending=False).model.iloc[0]
print(f"\nbest by PR-AUC: {best}")
print("\ncalibration:")
print(models.calibration_table(y, oof[best]).round(3).to_string(index=False))

               model  roc_auc  pr_auc  brier  prevalence    n
 baseline_prevalence    0.402   0.850  0.109       0.876 1009
logistic_descriptors    0.750   0.949  0.118       0.876 1009
       random_forest    0.822   0.964  0.091       0.876 1009

best by PR-AUC: random_forest

calibration:
 mean_predicted  observed_frequency
          0.423               0.525
          0.674               0.723
          0.759               0.802
          0.808               0.931
          0.843               0.931
          0.869               0.950
          0.893               0.970
          0.916               0.970
          0.935               0.980
          0.960               0.980


Two things to flag rather than gloss:

- **PR-AUC is inflated by prevalence.** At 87.6% positives, a constant predictor already scores
  0.850. ROC-AUC 0.822 against a 0.5 baseline is the more honest headline.
- **The model is systematically under-confident** — it predicts 0.42 where the observed rate is
  0.52 — and it **never predicts below ~0.42 at all**. So low-probability calibration simply cannot
  be assessed with this library. That is a direct argument for Part 3 deliberately buying compounds
  predicted *inactive*.

## 8. Questions we added

The prompt invites further questions. Three, each cheap because the data is already here.

### 8.1 Substrate versus inhibitor

All 1,223 reactivity compounds also have a CYP3A4 pIC50 in the inhibition release. Being *turned
over by* an enzyme and *inhibiting* it are different properties, and the dataset lets us check
whether they travel together.

In [13]:
inh = io.load_inhibition()
merged = df.merge(inh[["ocnt_batch", "CYP3A4_pIC50", "activity_status"]], on="ocnt_batch")
ok = merged[merged.call != "inconclusive"]
print(pd.crosstab(ok.call, ok.activity_status).to_string())
both = ok.dropna(subset=["CYP3A4_pIC50"])
r, p = st.spearmanr(both.CYP3A4_pIC50, both.delta_adj)
print(f"\nSpearman(pIC50, log10 FC) = {r:.3f}  (p={p:.2g}, n={len(both)})")
print("potent inhibitors (pIC50 > 6) that are confident NON-substrates:",
      int(((both.CYP3A4_pIC50 > 6) & (both.call == "non-substrate")).sum()))

activity_status  NO  YES
call                    
non-substrate    33   92
substrate        63  821

Spearman(pIC50, log10 FC) = -0.090  (p=0.0095, n=826)
potent inhibitors (pIC50 > 6) that are confident NON-substrates: 20


Essentially independent axes (|ρ| ≈ 0.09). 20 potent inhibitors (pIC50 > 6) are confident
non-substrates — compounds that bind and block without being metabolised. For DDI work that is the interesting
quadrant, and it is invisible if you treat "CYP activity" as one property.

### 8.2 CYP3A4 versus CYP2J2 selectivity

In [14]:
wide = calls.pivot_table(index="ocnt_batch", columns="enzyme", values="delta_adj").dropna()
sel = calls.pivot_table(index="ocnt_batch", columns="enzyme", values="call", aggfunc="first").dropna()
r, p = st.spearmanr(wide.CYP3A4, wide.CYP2J2)
print(f"Spearman(CYP3A4, CYP2J2 log10 FC) = {r:.3f}  (p={p:.2g}, n={len(wide)})\n")
print(pd.crosstab(sel.CYP3A4, sel.CYP2J2).to_string())

Spearman(CYP3A4, CYP2J2 log10 FC) = 0.243  (p=6.2e-18, n=1223)

CYP2J2         inconclusive  non-substrate  substrate
CYP3A4                                               
inconclusive             74            116         24
non-substrate            23             91         11
substrate               233            394        257


Only weakly correlated. 418 compounds are CYP3A4 substrates but confident CYP2J2 non-substrates,
while the reverse is rare — consistent with CYP3A4's famously permissive site versus CYP2J2's
narrower preference. The 3A4-selective set is a natural starting point for anyone wanting to probe
what CYP2J2 specifically excludes.

### 8.3 Artefact control — is "depletion" just weak signal?

The one that could invalidate everything. If apparent depletion tracked how strongly a compound
ionises, the hits would be an MS artefact rather than metabolism.

In [15]:
r2, p2 = st.spearmanr(df.mu_control, df.delta_adj)
print(f"Spearman(control signal strength, log10 FC) = {r2:.3f}  (p={p2:.2g})")
print("\nNo relationship: hits are not an artefact of how well a compound flies.")
fly = io.load_will_it_fly()
f2 = df.merge(fly, on="standardized_smiles")
if len(f2) > 20:
    r3, p3 = st.spearmanr(np.log10(f2.ammonium_fluoride_area.clip(lower=1)), f2.delta_adj)
    print(f"\n(overlap with the ionisation screen is only {len(f2)} compounds: rho={r3:.3f}, p={p3:.2g})")

Spearman(control signal strength, log10 FC) = -0.014  (p=0.63)

No relationship: hits are not an artefact of how well a compound flies.

(overlap with the ionisation screen is only 42 compounds: rho=0.138, p=0.38)


A flat, non-significant relationship is exactly what we want here. Had it been strongly negative,
the whole substrate list would need re-examining.

## 9. What Part 2 hands to Part 3

- **Confirmed cliffs** (572 significant pairs) — the highest-information regions of the SAR
  landscape, and where a similarity-based model is most likely to be wrong.
- **Implicated substructures** — associations that are correlational and need a causal test in
  fresh chemistry.
- **A scaffold-validated model**, with a documented calibration weakness at the low end.
- **A stated limitation**: effects are modest, scaffolds are underpowered, and the model is a
  ranker more than a calibrated probability.

Each of those becomes a purchasing bucket in Part 3.